# Interested in viewing the bathymetry or mesh that was used in the model to create the CORA data? This notebook allows users to create a rasterized plot of the topobathy at the CORA model nodes and overlay the model mesh.

*Parts of the code in this notebook were guided by **Rudiger, P. (2023). Bay Trimesh. https://github.com/holoviz/datashader/blob/f23de596f9adcb8188d48e6b163c36c913cd9912/examples/user_guide/6_Trimesh.ipynb#L11***

In [ ]:
import numpy as np
import requests
import matplotlib.pyplot as plt
import pandas as pd
import dask
import intake
import xarray as xr
import holoviews as hv
import datashader as ds
import datashader.transfer_functions as tf
import datashader.utils as du
import geoviews as gv
from holoviews import opts
import holoviews.operation.datashader as dshade
from holoviews.operation.datashader import datashade, shade, rasterize
import cmocean
hv.extension('bokeh')

**Access the data on the NODD and initialize the available CORA datasets.** 

*This accesses a .yml file located on the NODD that shows which CORA output files are available to import.*

In [ ]:

catalog = intake.open_catalog("s3://noaa-nos-cora-pds/CORA_V1.1_intake.yml",storage_options={'anon':True})
list(catalog)


**CORA-V1.1-fort.63: Hourly water levels <br>
CORA-V1.1-swan_DIR.63: Hourly mean wave direction <br>
CORA-V1.1-swan_TPS.63: Hourly peak wave periods <br>
CORA-V1.1-swan_HS.63: Hourly significant wave heights <br>
CORA-V1.1-Grid: Hourly water levels interpolated from model nodes to uniform 500-meter resolution grid <br>
All datasets denoted as timeseries are optimized for pulling long time series (greater than a few days) <br>
For up to a few days of data, use the regular dataset (not labeled timeseries)**

*Now, create an xarray dataset for the CORA data that you would like to use.*

In [ ]:
cora = catalog["CORA-V1.1-fort.63"].to_dask()


In [ ]:
def filter_mesh_to_region(verts_df, tris_df, bounds):
    """
    More mesh filtering using vectorized operations
    """
    # Create mask for vertices in region
    mask = (
        (verts_df['x'] >= bounds['lon_min']) &
        (verts_df['x'] <= bounds['lon_max']) &
        (verts_df['y'] >= bounds['lat_min']) &
        (verts_df['y'] <= bounds['lat_max'])
    )

    # Get vertices in region
    region_verts = verts_df[mask].copy().reset_index()

    # Create efficient index mapping using numpy
    old_indices = region_verts['index'].values
    new_indices = np.arange(len(region_verts))
    index_map = dict(zip(old_indices, new_indices))

    # Only keep triangles where ALL vertices are in the region
    valid_mask = np.isin(tris_df.values, old_indices).all(axis=1)
    region_tris = tris_df[valid_mask].copy()

    # Remap triangle indices
    for col in ['v0', 'v1', 'v2']:
        region_tris[col] = region_tris[col].map(index_map)

    # Remove the original index column
    region_verts = region_verts.drop('index', axis=1).reset_index(drop=True)
    region_tris = region_tris.reset_index(drop=True)

    return region_verts, region_tris

In [ ]:
# Create vertices and triangles from the full dataset
v = np.vstack((cora.x, cora.y, cora.depth)).T
verts_original = pd.DataFrame(v, columns=['x', 'y', 'z'])
tris_original = pd.DataFrame(cora['element'].values.astype('int')-1, columns=['v0','v1','v2'])

# Define Texas bounding box
texas_bounds = {'lon_min': -98.0, 'lon_max': -93.0, 'lat_min': 25.0, 'lat_max': 31.0}

# Use the efficient filtering function
verts, tris = filter_mesh_to_region(verts_original, tris_original, texas_bounds)

print(f"Original vertices: {len(verts_original)}, Texas coast vertices: {len(verts)}")
print(f"Original triangles: {len(tris_original)}, Texas coast triangles: {len(tris)}")

points = gv.operation.project_points(gv.Points(verts, vdims=['z']))

In [ ]:
base_url = 'https://api.tidesandcurrents.noaa.gov/mdapi/prod/webapi/stations/.json'
params = {
    'type': 'waterlevels',
    'units': 'metric'
}
print(f'base_url: {base_url}, parameters: {params}')
response = requests.get(base_url, params=params)
content = response.json()

stations = content['stations']
stations_df = pd.DataFrame(stations)

# Include station name along with id, lat, lng
stations_df = stations_df[['id','name','lat','lng','state']]

# limit to Texas stations
stations_df = stations_df[stations_df['state'] == 'TX']

# Filter for Texas stations
texas_stations = stations_df[stations_df['state'] == 'TX'].copy()

print(f"Found {len(texas_stations)} stations:")
texas_stations = texas_stations.reset_index(drop=True)
display(texas_stations)

In [ ]:
# Create station points for overlay
station_points = hv.Points(
    texas_stations,
    kdims=['lng', 'lat'],
    vdims=['name', 'id']
).opts(
    size=8,
    color='red',
    marker='circle',
    line_color='white',
    line_width=0.5,
    tools=['hover'],
    alpha=0.8
)



opts.defaults(
    opts.Image(width=1000, height=700),
    opts.RGB(width=1000, height=700))

# Create wireframe for mesh overlay
# the alpha value for the mesh lines is from 0 - 254
wireframe = datashade(hv.TriMesh((tris,verts)).edgepaths, alpha=64)

# Create the trimesh with depth values
trimesh = hv.TriMesh((tris, hv.Points(verts, vdims='z')))

dmesh = dshade.rasterize(trimesh).opts(cmap=cmocean.cm.deep, colorbar=True)

# Overlay the bathymetry, wireframe, and station points
dmesh * wireframe * station_points